In [ ]:
%load_ext autoreload

In [ ]:
from pathlib import Path
import re

import matplotlib.pyplot as plt
import mne
import pandas as pd
import torch
from tqdm.auto import tqdm
tqdm.pandas()

In [ ]:
%autoreload 2

from src.data import add_metadata_features
from src.models import causal4

In [ ]:
epochs_paths = list(Path("outputs/epochs_preprocessed").glob("*.fif"))

tg_dir = "textgrids"

timit_epoch_sources = {
    "All": "/userdata/jgauthier/projects/ideal-word-representations/phoneme_epochs-all.h5",
    "Word onset": "/userdata/jgauthier/projects/ideal-word-representations/phoneme_epochs-onsets.h5",
}

electrodes_paths = list(Path("outputs/causal4/find_speech_responsive").glob("*_results.csv"))

# all_A_result_paths = list(Path("outputs/causal4/find_As").glob("*_results.csv"))
# all_A_decoder_paths = list(Path("outputs/causal4/find_As").glob("*_decoders.pt"))
A_result_path = Path("outputs/causal4/unify_As/results.csv")
A_decoders_path = Path("outputs/causal4/unify_As/unified_decoders.pt")
all_B_result_paths = list(Path("outputs/causal4/find_Bs").glob("*_results.csv"))

alpha = 0.01

# Plot at most this many results
plot_max = 200

outdir = "outputs/causal4/plot_B_study"

## Load results

In [ ]:
epochs = {
    re.search(r"(\w+)_epo.fif", str(path)).group(1): mne.read_epochs(path, preload=True, verbose=False)
    for path in epochs_paths
}

In [ ]:
for e in epochs.values():
    e.metadata = add_metadata_features(e.metadata)

In [ ]:
electrode_df = pd.concat([pd.read_csv(path) for path in electrodes_paths]).set_index(["subject", "electrode_idx"])

In [ ]:
A_results = pd.read_csv(A_result_path)
B_results = pd.concat(
    [pd.read_csv(path) for path in all_B_result_paths],
    ignore_index=True,
)

In [ ]:
# Filtering on B results prior to plotting

# Exclude results where control p-value looks anywhere near good, or where it is better than the experimental p-value
B_results = B_results[B_results["stim_control_p_val_min"] > 0.05]
B_results = B_results[B_results.p_val_min < B_results.stim_control_p_val_min]

B_results = B_results[B_results.p_val_min < alpha].sort_values("p_val_min")
B_results = B_results.head(plot_max)

In [ ]:
A_decoders = torch.load(A_decoders_path)

In [ ]:
assert set(epochs.keys()) == set(A_decoders["train_scores"].subject.unique())
assert set(epochs.keys()) == set(A_results.subject)
assert set(epochs.keys()) == set(B_results.subject)
subjects = sorted(A_results.subject.unique())

## Plot and save

In [ ]:
plotter = causal4.Causal4Plotter(
    epochs=epochs,
    A_results=A_results,
    B_results=B_results,
    A_decoders=A_decoders,
    electrode_df=electrode_df,
    textgrid_dir=tg_dir,
    timit_epoch_sources=timit_epoch_sources,
)

In [ ]:
from matplotlib.backends.backend_pdf import PdfPages

with PdfPages(f"{outdir}/B_study.pdf") as pdf:
    for _, row in tqdm(B_results.iterrows(), total=len(B_results)):
        facetgrids = plotter(row)
        for fg in facetgrids:
            if fg is None:
                continue
            fg.tight_layout()
            fig = fg.fig if hasattr(fg, 'fig') else fg
            pdf.savefig(fig)
            plt.close(fig)

In [ ]:
B_results.to_csv(f"{outdir}/B_study_results.csv", index=False)